In [3]:
import altair as alt
import pandas as pd

# Load the data
df_fares = pd.read_csv("Quarterly national level domestic average fare series.csv")

# Inspect Data
df_fares.head()
df_fares.columns

### Nominal vs. inflation-adjusted airfare, 1995–2025

In [ ]:
# Data Preparation

def quarter_to_month(q):
    """Maps quarter number to the first month of that quarter."""
    if q == 1: return '01'
    if q == 2: return '04'
    if q == 3: return '07'
    if q == 4: return '10'
    return '01'

df_fares['Date'] = df_fares['Year'].astype(str) + '-' + df_fares['Quarter'].apply(quarter_to_month) + '-01'
df_fares['Date'] = pd.to_datetime(df_fares['Date'], format='%Y-%m-%d')

# Melt the data into 'long' format for Altair
df_long = df_fares.melt(
    id_vars=['Date'],
    value_vars=['Current', 'Inflation-Adjusted'],
    var_name='Fare Type',
    value_name='Average Fare ($)'
)

# Key Event Annotations
key_events = pd.DataFrame([
    {'date': '2001-10-01', 'event': '9/11 Attacks'}, 
    {'date': '2008-07-01', 'event': 'Oil Crisis Peak'}, 
    {'date': '2020-04-01', 'event': 'COVID-19 Collapse'} 
])
key_events['date'] = pd.to_datetime(key_events['date'], format='%Y-%m-%d')

# Altair Visualization
base = alt.Chart(df_long).encode(
    alt.X('Date:T', title='Year', axis=alt.Axis(format="%Y")),
    alt.Y('Average Fare ($):Q', title='Average Fare (USD)', scale=alt.Scale(domain=[250, 700])),
    color=alt.Color('Fare Type:N', title='Fare Type',
                    scale=alt.Scale(domain=['Current', 'Inflation-Adjusted'],
                                    range=['#E57373', '#4DB6AC'])) # Red for Current, Green for Real
).properties(
    title='U.S. Domestic Average Quarterly Airfare: Current vs. Inflation-Adjusted (1995–2025)'
)

# Line Chart Layer
line_chart = base.mark_line(point=False).encode(
    tooltip=[alt.Tooltip('Date:T', format='%Y-%m', title='Date'), 'Fare Type', alt.Tooltip('Average Fare ($):Q', format='$.2f')]
)

# Rule Annotations Layer
rules = alt.Chart(key_events).mark_rule(color='gray', strokeDash=[3, 3], size=1).encode(
    x='date:T',
    tooltip=[alt.Tooltip('date:T', format='%Y-%m-%d', title='Date'), 'event:N']
)

# Text Annotation Layer
text_labels = alt.Chart(key_events).mark_text(
    align='left', dx=5, dy=-5, angle=270, color='gray', fontSize=10
).encode(
    x='date:T',
    text='event:N',
    tooltip=[alt.Tooltip('date:T', format='%Y-%m-%d', title='Date'), 'event:N']
)

# Layer, make interactive, and save
chart = alt.layer(rules, line_chart, text_labels).interactive().properties(
    width=700,
    height=400
)

chart
#chart.save("current_vs_constant_airfare.json")
#chart.save("current.html")

alt.LayerChart(...)

### Findings

The dual-line chart above shows the central tension of modern air travel that we're examining: the perceived cost versus the real, inflation-adjusted value. While the Current Dollar (Nominal) Fare (the red line) shows a general upward trend (often giving travelers the impression that flights are constantly getting more expensive), the Inflation-Adjusted Fare (the green line) tells a different story:
Despite episodic spikes, the real cost of flying has declined significantly over the past three decades. Using Q1 1995 as a benchmark, the long-term trend shows a major decrease in what an airfare ticket costs relative to other consumer goods. This shows the impact of deregulation and increasing competition from low-cost carriers.

The chart also illustrates how major external shocks interrupt the long-term decline. We examined:

- 9/11 Attacks (2001): The initial drop in demand caused fares to plunge.

- 2008 Oil Crisis: This caused the most acute spike outside of the pandemic. As fuel costs hit record highs, the inflation-adjusted fare surged, forcing airlines to pass the expense directly to consumers.

- COVID-19 Pandemic (2020): Fares cratered to a historic low, demonstrating that even rock-bottom prices couldn't entice travelers during the global shutdown.

The overall takeaway for the traveler is that the cost of flying is not steadily increasing; it is an economic roller coaster where long-term affordability is periodically interrupted by massive, yet brief, external shocks.

### Airfare over time with year filter

In [ ]:
# Create a full date column for the X-axis (using the 1st of the month for the quarter)
df_fares['Date'] = pd.to_datetime(df_fares['Year'].astype(str) + '-' + (df_fares['Quarter'] * 3 - 2).astype(str) + '-01')

# Melt the DataFrame to long format 
df_long = df_fares.melt(
    id_vars=['Date', 'Year'], 
    value_vars=['Current', 'Inflation-Adjusted'], 
    var_name='Price Type', 
    value_name='Price ($)'
)

# Define the range of the slider using min and max years
min_year = int(df_long['Year'].min())
max_year = int(df_long['Year'].max())

year_slider = alt.binding_range(
    min=min_year,
    max=max_year,
    step=1,
    name='Filter by Year: '
)

# Apply the selection to a parameter
select_year = alt.selection_point(
    name="YearFilter",
    fields=['Year'],
    bind=year_slider,
    value=[{'Year': max_year}] # Default to the latest year
)

# Altair chart
base = alt.Chart(df_long).encode(
    x=alt.X('Date:T', title='Year'),
    y=alt.Y('Price ($):Q', title='Average Airfare ($)'),
    tooltip=[
        alt.Tooltip('Date:T', title='Quarter', format='%Y-Q%q'),
        alt.Tooltip('Price ($):Q', title='Price', format='$.2f'),
        'Price Type:N'
    ]
).properties(
    title='National Average Airfare: Current vs. Inflation-Adjusted Price'
)

# Line marks for the two series
lines = base.mark_line().encode(
    color=alt.Color('Price Type:N', title='Price Type', 
                    scale=alt.Scale(domain=['Current', 
                                            'Inflation-Adjusted'], 
                                            range=['#3366cc', '#cc0000'])),
    strokeDash=alt.condition(
        alt.datum['Price Type'] == 'Current', 
        alt.value([1, 0]),  # Solid line for Current
        alt.value([4, 4])   # Dashed line for Inflation-Adjusted
    )
).add_params(
    select_year # slider parameter
).transform_filter(
    select_year # slider filter 
)

chart_cpi_slider = lines
chart_cpi_slider

#chart_cpi_slider.save('airfare_vs_cpi_slider.json')
#chart_cpi_slider.save('airfare_vs_cpi_slider.html')

alt.Chart(...)

### Average airfare by city, 1995

Highlight individual cities to compare fares.

In [ ]:
# Load data 
df_city = pd.read_csv('AverageFare_Q1_1995.csv', skiprows=1) # skiprows=1 to skip the first row, index 0

# Select and rename columns
df_city_analysis = df_city[['City Name', 'State Name', 'Average Fare ($)']].copy()
df_city_analysis['City-State'] = df_city_analysis['City Name'] + ', ' + df_city_analysis['State Name']

# Clean the Fare column
df_city_analysis['Average Fare ($)'] = pd.to_numeric(
    df_city_analysis['Average Fare ($)'], 
    errors='coerce'
)

# Remove the summary "National Average" row and missing data
df_city_analysis = df_city_analysis.dropna(subset=['Average Fare ($)', 'City Name'])

# Sort data for proper ranking in the bar chart
df_city_analysis = df_city_analysis.sort_values('Average Fare ($)', ascending=False)

# Altair Chart
city_dropdown = alt.binding_select( # define dropdown
    options=df_city_analysis['City-State'].unique().tolist(),
    name='Select City: '
)
select_city = alt.selection_point(
    name="CitySelector", 
    fields=['City-State'], 
    bind=city_dropdown,
    # Set a default value to an "interesting" city (Boston)
    value=[{'City-State': 'Boston, MA'}] 
)

# Bar Chart
bars = alt.Chart(df_city_analysis).mark_bar().encode(
    # Rank the cities on the Y-axis by Fare. The y-axis to use the nominal city-state name.
    y=alt.Y('City-State:N', sort='x', axis=None), # hide axis labels for cleaner rank
    x=alt.X('Average Fare ($):Q', title='Average Fare ($) in Q1 1995'),
    
    # Tooltip for hover
    tooltip=['City-State:N', alt.Tooltip('Average Fare ($):Q', format='$.2f')],

    # Conditional color based on the dropdown selection
    color=alt.condition(
        select_city,
        alt.value('#1f77b4'),  # blue if selected
        alt.value('lightgray') # gray otherwise
    )
).properties(
    title='Ranked Average Airfares by City (Q1 1995)'
)

# Add a text label for showing the fare value next to the bar
text = bars.mark_text(
    align='left',
    baseline='middle',
    dx=3  #avoid overlap with the bar
).encode(
    text=alt.Text('Average Fare ($):Q', format='$.2f'),
    color=alt.condition(
        select_city,
        alt.value('black'),  # black text for selected
        alt.value('lightgray') # gray text for unselected
    )
)

# Combine the bars and text, add the selection, and make it interactive!
chart_city_comparison = alt.layer(bars, text).add_params(
    select_city
).interactive(
    # Allow zooming on the x-axis (fares)
    bind_x=True 
).properties(
    # Set a fixed width/height for better display
    width=600, 
    height=400
)
chart_city_comparison
# chart_city_comparison.save('city_fare_comparison.json')

alt.LayerChart(...)

#### Quarter-over-quarter airfare change during COVID-19, 2018–2022

Measuring and visualizing quarter-over-quarter percentage changes to highlight the magnitude and volatility of price shocks during the pandemic period.

In [ ]:
# Load data
df_fares = pd.read_csv('Quarterly national level domestic average fare series.csv') 

# Create the correct Date column
df_fares['Date'] = pd.to_datetime(
    df_fares['Year'].astype(str) + '-' + (df_fares['Quarter'] * 3 - 2).astype(str) + '-01'
)
# Filter for COVID-19 period (2018–2022)
df_covid = df_fares[(df_fares['Year'] >= 2018) & (df_fares['Year'] <= 2022)].copy()
df_covid = df_covid.sort_values('Date').reset_index(drop=True)

# Calculate the Quarter-over-Quarter (QoQ) percentage change in Current fare
df_covid['QoQ Change (%)'] = df_covid['Current'].pct_change() * 100

# Remove the first row which will have NaN
df_change = df_covid.dropna(subset=['QoQ Change (%)']).copy()

#Altair Bar Chart of Change
bar_chart = alt.Chart(df_change).mark_bar().encode(
    x=alt.X('Date:T', title='Quarter'),
    y=alt.Y('QoQ Change (%):Q', title='Quarter-over-Quarter Change in Fare (%)'),
    
    # Conditional coloring to highlight the collapse
    color=alt.condition(
        alt.datum['QoQ Change (%)'] < 0,
        alt.value('red'),  # negative changes are red (the collapse)
        alt.value('darkgreen') # positive changes are green (the rebound)
    ),
    tooltip=[
        alt.Tooltip('Date:T', title='Quarter', format='%Y-Q%q'),
        alt.Tooltip('QoQ Change (%):Q', title='QoQ Change', format='.1f'),
        alt.Tooltip('Current:Q', title='Fare ($)', format='$.2f')
    ]
).properties(
    title='Impact of COVID-19: Quarterly Airfare Percentage Change (2018–2022)',
    width=700,
    height=350
)

# Add a zero line for context
zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(strokeDash=[4, 4], color='black').encode(y='y')

chart_qoq = (bar_chart + zero_line).interactive()
chart_qoq.save('covid_qoq_change_bar.json')
chart_qoq

alt.LayerChart(...)

#### Findings

The most interesting feature of the chart is the deep-red bar corresponding to the second quarter of 2020 (Q2 2020). This bar represents the largest quarterly decline in average domestic airfare in the dataset's history. The average domestic airfare dropped by approximately 19.8% in Q2 2020 compared to Q1 2020. This is the quarter when most travel restrictions were enacted. Following the massive negative spike, the chart shows a pattern of extreme volatility, characterized by alternating large green (positive) and red (negative) bars in 2021 and 2022. This is possibly due to the market's erratic attempts to find a new equilibrium as demand pulsed back and forth. The bars representing the years 2018 and 2019 are relatively small and centered around the zero line, demonstrating the pre-pandemic stability of the market. The dramatic scale of the 2020 drop and subsequent rebound starkly highlights the event as an unprecedented external shock to the airline economy.

In [43]:
import pandas as pd
import altair as alt

# --- 1. Load and Prepare Data ---
df_fares = pd.read_csv('Quarterly national level domestic average fare series.csv') 
df_fares['Date'] = pd.to_datetime(
    df_fares['Year'].astype(str) + '-' + (df_fares['Quarter'] * 3 - 2).astype(str) + '-01'
)

# --- 2. Define the Pre-Pandemic Baseline (Q4 2019) ---
# FIX: Explicitly convert to float for safe Altair/JSON substitution.
baseline_fare = float(df_fares[
    (df_fares['Year'] == 2019) & (df_fares['Quarter'] == 4)
]['Inflation-Adjusted'].iloc[0]) 

baseline_df = pd.DataFrame({'baseline': [baseline_fare]})

# Filter data to focus on the post-2018 period for better visual impact
df_surge = df_fares[df_fares['Year'] >= 2019]

# --- 3. Altair Chart Construction ---

# Reference Line at the Q4 2019 Inflation-Adjusted Price
reference_line = alt.Chart(baseline_df).mark_rule(
    strokeDash=[5, 5], 
    color='darkgray'
).encode(
    y=alt.Y('baseline:Q')
)

# Area Chart for the Surge (fills the space above the baseline)
area_surge = alt.Chart(df_surge).mark_area(
    clip=True,
    opacity=0.3,
    color='orange'
).encode(
    x='Date:T',
    y=alt.Y('Inflation-Adjusted:Q', scale=alt.Scale(domain=[300, 480])), 
    y2=alt.Y2(datum=baseline_fare) # y2 is anchored by the literal float value
).transform_filter(
    alt.datum['Inflation-Adjusted'] >= baseline_fare # Filter is anchored by the literal float value
)

# Main Line Chart (Inflation-Adjusted)
line_chart = alt.Chart(df_surge).mark_line(color='#cc0000').encode(
    x=alt.X('Date:T', title='Year'),
    y=alt.Y('Inflation-Adjusted:Q', title='Inflation-Adjusted Fare (USD)'),
    tooltip=[
        alt.Tooltip('Date:T', format='%Y-Q%q', title='Quarter'),
        alt.Tooltip('Inflation-Adjusted:Q', format='$.2f', title='Real Fare')
    ]
)

# Text Annotation for Baseline
text_label = alt.Chart(baseline_df).mark_text(
    align='left', 
    dx=10, 
    dy=-5, 
    color='darkgray', 
    fontSize=11,
    text=f'Pre-COVID Real Fare Baseline (${baseline_fare:,.2f})'
).encode(
    y=alt.Y('baseline:Q')
)

# Final Layering
chart_surge = (area_surge + reference_line + line_chart + text_label).interactive().properties(
    title='The Real Cost Surge: How Post-Pandemic Fares Exceeded Pre-COVID Levels'
)

chart_surge

alt.LayerChart(...)

### Airfare and jet fuel prices, 1995–present

In [7]:
import pandas as pd

df_raw = pd.read_excel("EER_EPJK_PF4_RGC_DPGm.xls", header=None)
df_raw.head()

In [8]:
df_air_raw = pd.read_excel("Quarterly national level domestic average fare series.xls", header=None)
df_air_raw.head()

In [ ]:
# Inspect sheet names
xls = pd.ExcelFile("EER_EPJK_PF4_RGC_DPGm.xls")
xls.sheet_names

In [ ]:
# Load the jet fuel price data from the specified sheet
df_fuel_raw = pd.read_excel("EER_EPJK_PF4_RGC_DPGm.xls",
    sheet_name="Data 1",
    header=None
)

# Clean and prepare the jet fuel data
df_fuel = df_fuel_raw.copy()
df_fuel = df_fuel[df_fuel[0].astype(str).str.contains("199", na=False)]

# Rename columns
df_fuel = df_fuel.rename(columns={0: "Date", 1: "JetFuelPrice"})

# Convert Date to datetime
df_fuel["Date"] = pd.to_datetime(df_fuel["Date"])

# Convert price to numeric
df_fuel["JetFuelPrice"] = pd.to_numeric(df_fuel["JetFuelPrice"], errors="coerce")

# Drop any missing rows
df_fuel = df_fuel.dropna()

df_fuel.head()


In [ ]:

# Load Jet Fuel Monthly Data

df_fuel_raw = pd.read_excel(
    "EER_EPJK_PF4_RGC_DPGm.xls",
    sheet_name="Data 1",
    header=None
)

# Identify the real data rows (start when the first date appears)
df_fuel = df_fuel_raw[df_fuel_raw[0].astype(str).str.contains("199", na=False)].copy()

# Clean columns
df_fuel = df_fuel.rename(columns={0: "Date", 1: "JetFuelPrice"})
df_fuel["Date"] = pd.to_datetime(df_fuel["Date"])
df_fuel["JetFuelPrice"] = pd.to_numeric(df_fuel["JetFuelPrice"], errors="coerce")

df_fuel = df_fuel.dropna()

# -------------------------------
# Convert monthly → quarterly average
# -------------------------------
df_fuel_quarterly = (
    df_fuel.set_index("Date")
           .resample("Q")      # quarter-end
           .mean()
           .reset_index()
)

df_fuel_quarterly.head()


In [96]:
import pandas as pd
import altair as alt

# Load airfare from the CLEAN CSV version
df_air = pd.read_csv("Quarterly national level domestic average fare series.csv")

# Extract Year and Quarter from CSV
df_air["Year"] = df_air["Year"].astype(int)
df_air["Quarter"] = df_air["Quarter"].astype(int)

# Keep current-dollar fares
df_air = df_air[["Year", "Quarter", "Current"]].rename(columns={"Current": "Airfare"})

df_air.head(3)


In [97]:
# Load jet fuel monthly data
df_fuel_raw = pd.read_excel(
    "EER_EPJK_PF4_RGC_DPGm.xls",
    sheet_name="Data 1",
    header=None
)

# Keep only rows with dates like "1990-04-15 ..."
df_fuel = df_fuel_raw[df_fuel_raw[0].astype(str).str.contains("19", na=False)].copy()

# Clean columns
df_fuel = df_fuel.rename(columns={0: "Date", 1: "JetFuelPrice"})
df_fuel["Date"] = pd.to_datetime(df_fuel["Date"])
df_fuel["JetFuelPrice"] = pd.to_numeric(df_fuel["JetFuelPrice"], errors="coerce")
df_fuel = df_fuel.dropna()

# Extract Year + Quarter
df_fuel["Year"] = df_fuel["Date"].dt.year
df_fuel["Quarter"] = df_fuel["Date"].dt.quarter

# Aggregate monthly → quarterly mean
df_fuel_q = (
    df_fuel.groupby(["Year", "Quarter"])["JetFuelPrice"]
    .mean()
    .reset_index()
)

df_fuel_q.head()

In [98]:
df_merged = pd.merge(
    df_air,
    df_fuel_q,
    on=["Year", "Quarter"],
    how="inner"
)

df_merged.head()


In [99]:
# Normalize both series to index values (1995 Q1 = 100)
base_air = df_merged["Airfare"].iloc[0]
base_fuel = df_merged["JetFuelPrice"].iloc[0]

df_merged["Airfare_Index"] = (df_merged["Airfare"] / base_air) * 100
df_merged["Fuel_Index"] = (df_merged["JetFuelPrice"] / base_fuel) * 100

df_merged.head()


In [104]:
import altair as alt

# Assuming your data is in df_merged
df_plot = df_merged.copy()
# Ensure time is correctly formatted for ordering/coloring
df_plot["Time"] = pd.to_datetime(df_plot["Year"].astype(str) + "-Q" + df_plot["Quarter"].astype(str))

# --- 1. Base Chart Setup ---
base = alt.Chart(df_plot).properties(
    title="The Price Shock Path: How Airfares Respond to Fuel Costs (1995–Present)",
    width=600,
    height=450
).encode(
    # X-Axis: The Cost Driver
    x=alt.X("Fuel_Index:Q", title="Jet Fuel Price Index (Base Q1 1995 = 100)", scale=alt.Scale(zero=False)),
    # Y-Axis: The Retail Price
    y=alt.Y("Airfare_Index:Q", title="Domestic Airfare Index (Base Q1 1995 = 100)", scale=alt.Scale(zero=False)),
    # Tooltip: Shows all original and indexed values for rich interaction
    tooltip=[
        "Time:T",
        alt.Tooltip("Airfare_Index:Q", title="Airfare Index", format=".1f"),
        alt.Tooltip("Fuel_Index:Q", title="Fuel Index", format=".1f"),
        alt.Tooltip("Airfare:Q", title="Airfare ($)", format=".2f"),
        alt.Tooltip("JetFuelPrice:Q", title="Fuel ($/gal)", format=".3f"),
    ]
)

# --- 2. Sequential Path Line ---
path_line = base.mark_line(color='gray', opacity=0.6).encode(
    order="Time:T" # Essential for connecting points in chronological order
)

# --- 3. Points Colored by Time ---
points = base.mark_circle(size=80).encode(
    # Color by Year/Time to show progression along the path
    color=alt.Color("Time:T", scale=alt.Scale(scheme="viridis"), title="Year"),
)

# --- 4. Linear Trend Line (Context for Correlation) ---
trend_line = base.transform_regression("Fuel_Index", "Airfare_Index").mark_line(color="black", strokeDash=[5,5])

# --- 5. Combine and Finalize ---
final_chart = (path_line + points + trend_line).interactive()

# Save the interactive chart specification
#final_chart.save('price_shock_path_interactive.json')

final_chart

alt.LayerChart(...)

This visualization shows how closely domestic airfares have tracked jet fuel prices over the past 30 years—and where that relationship has broken down. In the early years of the series, when fuel was cheap and relatively stable, airfares clustered tightly as well, reflecting a period of intense airline competition and steady consumer demand. The path then swings sharply to the right in the mid-2000s, during the global oil shock, when jet fuel prices surged far faster than ticket prices—evidence that airlines absorbed much of the hit through fees, capacity cuts, and operational changes rather than passing the full cost to travelers. The steep downward drop marks the COVID-19 collapse, when both fuel costs and fares plunged simultaneously to record lows. And on the far right edge, the line bends upward again during the 2022–23 rebound, as high fuel prices and a surge in post-pandemic travel pushed fares to their highest levels in years. Taken together, the chart reveals a long-term dance between fuel costs and ticket prices, punctuated by economic shocks that pull the two series apart before they fall back into step.